# 01 -- Behavioral Validation

Analyze model behavioral results post-extraction: accuracy rates, lure susceptibility,
per-category performance, and reasoning model thinking trace statistics.

**What this notebook does:**
1. Load behavioral outcomes from the activation HDF5
2. Per-model accuracy and lure rates
3. Per-category behavioral breakdown
4. Standard vs reasoning model comparison
5. Thinking trace length analysis (reasoning models)
6. Response category distribution

**Modes:**
- **Synthetic** (default): uses `build_synthetic_hdf5` -- runs without GPU
- **Real**: change `DATA_PATH` to point to real activations

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from s1s2.utils.io import (
    open_activations, load_problem_metadata, list_models,
    model_metadata, get_behavior, get_generations, n_problems,
)
from s1s2.viz.theme import set_paper_theme, COLORS, MODEL_COLORS, get_model_color

set_paper_theme()

In [ ]:
# ---- Mode selection ----
# Synthetic mode (default): builds a small HDF5 for development
# Real mode: set DATA_PATH to your extraction output

USE_SYNTHETIC = True  # Set to False for real data

if USE_SYNTHETIC:
    from tests.conftest import build_synthetic_hdf5
    DATA_PATH = REPO_ROOT / "data" / "activations" / "synthetic_nb01.h5"
    build_synthetic_hdf5(DATA_PATH)
    print(f"Built synthetic HDF5 at {DATA_PATH}")
else:
    DATA_PATH = REPO_ROOT / "data" / "activations" / "main.h5"  # REQUIRES REAL DATA
    print(f"Using real data at {DATA_PATH}")

In [ ]:
# --- Load all behavioral data ---
with open_activations(DATA_PATH) as f:
    meta = load_problem_metadata(f)
    models = list_models(f)
    n_prob = n_problems(f)

    all_behavior = {}
    all_generations = {}
    all_model_meta = {}
    for m in models:
        all_behavior[m] = get_behavior(f, m)
        all_generations[m] = get_generations(f, m)
        all_model_meta[m] = model_metadata(f, m)

print(f"Models: {models}")
print(f"Problems: {n_prob}")
print(f"Conflict items: {meta['conflict'].sum()}")
print(f"Control items: {(~meta['conflict']).sum()}")

## Per-Model Accuracy and Lure Rates

In [ ]:
conflict = meta["conflict"]

rows = []
for m in models:
    beh = all_behavior[m]
    is_reasoning = all_model_meta[m].get("is_reasoning_model", False)
    rows.append({
        "model": m,
        "reasoning": is_reasoning,
        "acc_overall": beh["correct"].mean(),
        "acc_conflict": beh["correct"][conflict].mean(),
        "acc_control": beh["correct"][~conflict].mean(),
        "lure_rate_conflict": beh["matches_lure"][conflict].mean(),
        "lure_rate_overall": beh["matches_lure"].mean(),
    })

df_summary = pd.DataFrame(rows)
display(df_summary.style.format({
    "acc_overall": "{:.1%}",
    "acc_conflict": "{:.1%}",
    "acc_control": "{:.1%}",
    "lure_rate_conflict": "{:.1%}",
    "lure_rate_overall": "{:.1%}",
}).set_caption("Behavioral summary per model"))

In [ ]:
# Grouped bar chart: accuracy on conflict vs control
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x = np.arange(len(models))
w = 0.35

# Accuracy
axes[0].bar(x - w / 2, df_summary["acc_conflict"], w,
            label="Conflict", color=COLORS["s1"])
axes[0].bar(x + w / 2, df_summary["acc_control"], w,
            label="Control", color=COLORS["s2"])
axes[0].set_xticks(x)
axes[0].set_xticklabels([m.split("_")[-1][:15] for m in models], rotation=30, ha="right")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy: conflict vs control")
axes[0].set_ylim(0, 1)
axes[0].legend()
axes[0].axhline(0.5, color="gray", linestyle=":", alpha=0.5)

# Lure rate
axes[1].bar(x, df_summary["lure_rate_conflict"], color=COLORS["s1"], alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels([m.split("_")[-1][:15] for m in models], rotation=30, ha="right")
axes[1].set_ylabel("Lure rate (on conflict items)")
axes[1].set_title("S1-lure susceptibility")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## Per-Category Performance

In [ ]:
categories = np.unique(meta["category"])

cat_rows = []
for m in models:
    beh = all_behavior[m]
    for cat in categories:
        cat_mask = meta["category"] == cat
        cat_rows.append({
            "model": m.split("_")[-1][:15],
            "category": cat,
            "accuracy": beh["correct"][cat_mask].mean(),
            "lure_rate": beh["matches_lure"][cat_mask & conflict].mean()
                         if (cat_mask & conflict).any() else 0.0,
            "n": int(cat_mask.sum()),
        })

df_cat = pd.DataFrame(cat_rows)

# Heatmap of accuracy by model x category
pivot = df_cat.pivot(index="model", columns="category", values="accuracy")
fig, ax = plt.subplots(figsize=(8, max(3, len(models))))
im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i, j]:.0%}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax, label="Accuracy")
ax.set_title("Accuracy by model and category")
plt.tight_layout()
plt.show()

## Response Category Distribution

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 4), squeeze=False)
axes = axes[0]

for idx, m in enumerate(models):
    beh = all_behavior[m]
    cats_resp = pd.Series(beh["response_category"]).value_counts()
    axes[idx].pie(
        cats_resp.values,
        labels=[f"{k}\n({v})" for k, v in cats_resp.items()],
        autopct="%1.0f%%",
        colors=[COLORS["s2"], COLORS["s1"], COLORS["baseline"], COLORS["non_significant"]][:len(cats_resp)],
    )
    axes[idx].set_title(m.split("_")[-1][:15], fontsize=10)

plt.suptitle("Response category distribution", fontsize=12)
plt.tight_layout()
plt.show()

## Thinking Trace Statistics (Reasoning Models)

In [ ]:
reasoning_models = [m for m in models if all_model_meta[m].get("is_reasoning_model", False)]

if not reasoning_models:
    print("No reasoning models in this dataset (expected in synthetic mode).")
    print("With real data, this cell will show thinking trace length distributions.")
else:
    fig, axes = plt.subplots(1, len(reasoning_models),
                             figsize=(5 * len(reasoning_models), 4), squeeze=False)
    axes = axes[0]
    for idx, m in enumerate(reasoning_models):
        gen = all_generations[m]
        think_lens = gen["thinking_token_count"]
        # Split by conflict
        axes[idx].hist(think_lens[conflict], bins=20, alpha=0.7,
                       label="Conflict", color=COLORS["s1"])
        axes[idx].hist(think_lens[~conflict], bins=20, alpha=0.7,
                       label="Control", color=COLORS["s2"])
        axes[idx].set_xlabel("Thinking tokens")
        axes[idx].set_ylabel("Count")
        axes[idx].set_title(f"{m.split('_')[-1][:15]}")
        axes[idx].legend()
        # Stats
        print(f"{m}:")
        print(f"  Conflict thinking: mean={think_lens[conflict].mean():.0f}, "
              f"median={np.median(think_lens[conflict]):.0f}")
        print(f"  Control thinking:  mean={think_lens[~conflict].mean():.0f}, "
              f"median={np.median(think_lens[~conflict]):.0f}")

    plt.suptitle("Thinking trace length: conflict vs control", fontsize=12)
    plt.tight_layout()
    plt.show()

## Example Generations

In [ ]:
# Show a few example generations
m = models[0]
gen = all_generations[m]
beh = all_behavior[m]

print(f"Example generations from {m}:\n")
for i in range(min(3, n_prob)):
    cat_label = "CONFLICT" if conflict[i] else "CONTROL"
    correct_label = "CORRECT" if beh['correct'][i] else "WRONG"
    print(f"--- Item {i} [{cat_label}] [{correct_label}] ---")
    print(f"  Category: {meta['category'][i]}")
    print(f"  Predicted: {beh['predicted_answer'][i]}")
    print(f"  Expected: {meta['correct_answer'][i]}")
    answer = gen["answer_text"][i]
    print(f"  Answer text: {answer[:200]}")
    print()

## Summary

**Key behavioral patterns to verify before proceeding to mechanistic analysis:**

1. **Conflict effect**: accuracy should be lower on conflict items than controls across all models
2. **Lure susceptibility**: standard models should show higher S1-lure rates than reasoning models
3. **Reasoning traces**: reasoning models should produce longer thinking traces on conflict items (if they genuinely deliberate more)
4. **No floor/ceiling effects**: if a model is at chance or at 100%, probing will be uninformative

These behavioral patterns are necessary (but not sufficient) conditions for the mechanistic hypotheses tested in notebooks 02-05.